In [ ]:
# ===== 1) IMPORTS & GLOBAL CONFIG =====
from __future__ import annotations

import re
import time
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ---- What to scrape (edit these) ----
SCHOOLS = {
    # Use {year} in each URL; keep ?view=2 for the table/grid view when available.
    "Jackson State": "https://gojsutigers.com/sports/football/roster/{year}?view=2",
    "Alabama State": "https://bamastatesports.com/sports/football/roster/{year}?view=2",
    "Alabama A&M": "https://aamusports.com/sports/football/roster/{year}?view=2",
    "Southern": "https://gojagsports.com/sports/football/roster/{year}?view=2",
    "Prairie View A&M": "https://pvpanthers.com/sports/football/roster/{year}?view=2",
    "Texas Southern": "https://tsusports.com/sports/football/roster/{year}?view=2",
    "UAPB": "https://uapblionsroar.com/sports/football/roster/{year}?view=2",
    "Alcorn State": "https://alcornsports.com/sports/football/roster/{year}?view=2",
    "Grambling": "https://gsutigers.com/sports/football/roster/{year}?view=2",
    "Mississippi Valley State": "https://mvsusports.com/sports/football/roster/{year}?view=2",
    "Florida A&M": "https://famuathletics.com/sports/football/roster/{year}?view=2",
    "Bethune-Cookman": "https://bcuathletics.com/sports/football/roster/{year}?view=2"

}

YEARS = list(range(2010, 2026))     # inclusive range you want to scrape
HEADLESS = True                     # set False to watch the browser work
PAGE_LOAD_TIMEOUT = 20              # seconds to wait for page/table
POLITE_DELAY = 1.5                  # seconds between requests (be nice)
EMPTY_ROW_THRESHOLD = 1              # 0–1 row = “empty”
AUTO_STOP_CONSECUTIVE = 4            # stop after 4 empty years in a row

In [ ]:
# ===== 2) SELENIUM DRIVER FACTORY =====
def make_driver(headless: bool = True): # -> webdriver.Chrome:
    opts = Options() # browser options
    if headless: # run in headless mode (no GUI)
        opts.add_argument("--headless=new") # updated for Chrome 109+
    opts.add_argument("--no-sandbox") # required when running as root user. otherwise you would get no sandbox errors
    opts.add_argument("--disable-dev-shm-usage") # overcome limited resource problems
    opts.add_argument("--window-size=1920,1080") # set window size to avoid potential issues
    # Selenium Manager will fetch the right ChromeDriver automatically.
    driver = webdriver.Chrome(options=opts) # start the browser
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT) # set timeout
    return driver # return the driver


In [ ]:
# ===== 3) SCRAPER (ONE PAGE) =====
def scrape_roster_table(driver, url: str) -> pd.DataFrame | None: # -> pd.DataFrame | None:
    """Load Sidearm roster ?view=2 table and return a DataFrame or None."""
    try:
        driver.get(url) # load the page
    except Exception as e: # catch any loading errors
        print(f"[!] Page load failed: {url}\n    {e}") # log the error
        return None # return None on failure

    try: # wait for the roster table to load
        WebDriverWait(driver, PAGE_LOAD_TIMEOUT).until(  # wait up to timeout seconds
            EC.presence_of_element_located((By.CSS_SELECTOR, "table.sidearm-table")) # look for the table
        )
        time.sleep(0.75) # wait a bit more for stability
    except Exception: # if the table doesn't load in time
        print(f"[!] No roster table detected: {url}") # log the issue
        return None # return None if no table

    soup = BeautifulSoup(driver.page_source, "lxml") # parse the page source
    table = soup.find("table", class_="sidearm-table") # find the roster table
    if not table: # if no table found
        return None # return None

    thead = table.find("thead") # find the table header
    headers = [th.get_text(" ", strip=True) for th in thead.find_all("th")] if thead else [] # extract headers

    tbody = table.find("tbody") # find the table body
    if not tbody: # if no body found
        return None # return None

    rows = [] # list to hold row data
    for tr in tbody.find_all("tr"): # iterate over table rows
        tds = tr.find_all("td") # find all cells in the row
        if not tds: # if no cells found
            continue # skip this row
        rows.append([td.get_text(" ", strip=True) for td in tds]) # extract cell text

    if not rows: # if no rows found
        return None # return None

    df = pd.DataFrame(rows) # create DataFrame from rows
    if len(headers) == df.shape[1]: # if header count matches column count
        df.columns = headers # set DataFrame columns
    return df # return the DataFrame


In [ ]:
# ===== 4) NORMALIZATION HELPERS (fixed for dup columns + concat error) =====
HEADER_MAP = {
    "full name": "name", "name": "name", 
    "pos.": "pos", "position": "pos",
    "ht.": "ht", "height": "ht",
    "wt.": "wt", "weight": "wt",
    "yr.": "yr", "year": "yr",
    "#": "number", "no.": "number",
    "image": "image",
    # Combined columns used by various Sidearm sites:
    "hometown/previous school": "home_prev",
    "hometown / previous school": "home_prev",
    "hometown/high school": "home_prev",
    "hometown / high school": "home_prev",
}

HS_PATTERNS = re.compile( 
    r"(?:\bHS\b|\bHigh School\b|\bPrep\b|\bAcademy\b|\bSchool\b)$",
    re.IGNORECASE,
)

def canonicalize_headers(df: pd.DataFrame) -> pd.DataFrame:
    """Map different spellings to consistent names. Do NOT drop dups here."""
    df = df.copy()
    mapped = []
    for c in df.columns:
        key = re.sub(r"\s+", " ", c.strip().lower())
        mapped.append(HEADER_MAP.get(key, c))
    df.columns = mapped
    return df

def split_hometown_prev(s: str):
    """Return (hometown, state, high_school, previous_schools) from 'City, ST / School...'."""
    if not isinstance(s, str) or not s.strip():
        return None, None, None, None

    parts = [p.strip(" -\u2013\u2014").strip() for p in s.split("/") if p.strip()] # split on /
    city_state = parts[0] if parts else "" # first part is city/state
    hometown, state = None, None # initialize
    if "," in city_state: # split city/state
        left, right = city_state.split(",", 1) # split on first comma
        hometown = left.strip() # city
        state = right.strip() # state
    else: 
        hometown = city_state.strip() # no state provided

    schools = parts[1:] if len(parts) > 1 else [] # remaining parts are schools
    high_school, previous_schools = None, None # initialize
    if schools: # if there are schools listed
        last = schools[-1] # last school
        if HS_PATTERNS.search(last): # if it looks like a high school
            high_school = last # assign as high school
            prior = schools[:-1] # previous schools are the rest
        else: # otherwise
            prior = schools # all are previous schools
        if prior: # if there are previous schools
            previous_schools = " / ".join(prior) # join them with /

    return hometown or None, state or None, high_school, previous_schools

def expand_home_prev(df: pd.DataFrame) -> pd.DataFrame: # -> pd.DataFrame:
    """
    Safely expand combined column into separate parts. 
    Handles duplicate 'home_prev' columns by coalescing row-wise.
    """
    if "home_prev" not in df.columns: # if no combined column, return as-is
        return df # no changes needed

    out = df.copy() # work on a copy

    # All columns that are exactly 'home_prev' (could be duplicates after mapping)
    hp_mask = out.columns == "home_prev" # boolean mask
    hp_cols = list(out.columns[hp_mask]) # list of those columns

    if len(hp_cols) == 1: # only one column, no dup issue
        hp_series = out["home_prev"] # just use it directly
    else:
        # Coalesce across duplicates: take first non-empty cell per row
        hp_df = out.loc[:, hp_mask] # DataFrame of just those columns
        def pick_first_nonempty(row): # helper to pick first non-empty
            for val in row: # iterate over values
                if pd.notna(val) and str(val).strip(): # if non-empty
                    return str(val).strip() # return it
            return "" # if all empty, return empty string
        hp_series = hp_df.apply(pick_first_nonempty, axis=1) # apply row-wise

        # Drop extras, keep a single logical 'home_prev'
        to_drop = hp_cols[1:] # columns to drop
        out = out.drop(columns=to_drop) # drop them
        out.loc[:, "home_prev"] = hp_series # set the coalesced series

    hp_clean = hp_series.where(hp_series.notna(), "") # replace NaN with empty string
    parsed = hp_clean.map(split_hometown_prev) # parse each entry

    # Coerce anything odd to a 4-tuple
    def _coerce4(x): # -> tuple:
        if isinstance(x, (list, tuple)) and len(x) == 4: # already a 4-tuple
            return tuple(x) # return as-is
        try: # try to coerce
            lst = list(x) # convert to list
            return tuple(lst[:4]) + (None,) * (4 - len(lst)) # pad to 4
        except Exception: # on error
            return (None, None, None, None) # return all None

    parts = [_coerce4(p) for p in parsed] # ensure all are 4-tuples

    parts_df = pd.DataFrame( 
        parts,
        index=out.index,
        columns=["hometown", "state", "high_school", "previous_schools"],
    )

    out = pd.concat([out, parts_df], axis=1) # add new columns
    return out # return the expanded DataFrame

def normalize_roster(df: pd.DataFrame) -> pd.DataFrame: # -> pd.DataFrame:
    """
    Unify column names, expand combined hometown/prev-school, ensure unique labels,
    and return a consistent schema for concat.
    """
    df = canonicalize_headers(df).copy() # unify headers

    # If an explicit HS column exists, keep it
    hs_like = [c for c in df.columns if c.strip().lower() in {"high school", "hs"}]
    if hs_like:
        df.loc[:, "high_school"] = df[hs_like[0]]

    # Expand combined column (handles duplicate 'home_prev' internally)
    df = expand_home_prev(df)

    # Ensure unique column labels BEFORE selecting final columns
    df = df.loc[:, ~df.columns.duplicated()].copy()

    # Ensure high_school exists
    if "high_school" not in df.columns: 
        df["high_school"] = pd.NA # add empty HS column if missing

    # Final, consistent schema
    preferred_cols = [ 
        "number", "name", "pos", "ht", "wt", "yr", 
        "hometown", "state", "high_school", "previous_schools", 
    ]
    for c in preferred_cols: 
        if c not in df.columns: 
            df[c] = pd.NA

    return df[preferred_cols].copy()




In [ ]:
# ===== 5) MAIN LOOP: DESC + AUTO-STOP =====
def scrape_schools_years_desc_autostop(schools: dict[str,str], years: list[int]) -> pd.DataFrame: # -> pd.DataFrame:
    driver = make_driver(HEADLESS) # create the Selenium driver
    all_frames = [] # list to hold all school DataFrames
    try: # ensure driver quits
        for school, template in schools.items(): # iterate over schools
            print(f"\n=== {school} ===") # log the school
            school_frames = [] # list to hold this school's DataFrames
            consecutive_empty = 0 # counter for consecutive empty years

            for yr in sorted(years, reverse=True):  # newest → oldest
                url = template.format(year=yr) # fill in the year
                print(f"  Year {yr}: {url}") # log the year and URL

                df = scrape_roster_table(driver, url) # scrape the table
                if df is None or df.shape[0] <= EMPTY_ROW_THRESHOLD: # if empty or too few rows
                    consecutive_empty += 1 # increment counter
                    print(f"    -> Skipped (empty or <= {EMPTY_ROW_THRESHOLD} row). " # log skip
                          f"[{consecutive_empty} empty in a row]") # log skip
                else: # valid data found
                    consecutive_empty = 0 # reset counter
                    norm = normalize_roster(df).copy() # normalize
                    norm.loc[:, "school"] = school # add school column
                    norm.loc[:, "year"] = yr # add year column
                    school_frames.append(norm) # add to this school's list
                    print(f"    -> Got {norm.shape[0]} rows.") # log success

                # Auto-stop once we’ve seen many empty years consecutively
                if consecutive_empty >= AUTO_STOP_CONSECUTIVE: # if too many empty in a row
                    print(f"    -> Auto-stop for {school} " # log auto-stop
                          f"(no real data last {AUTO_STOP_CONSECUTIVE} years).") # log auto-stop
                    break # break out of year loop

                time.sleep(POLITE_DELAY) # polite delay between requests

            if school_frames: # if we got any data for this school
                school_df = pd.concat(school_frames, ignore_index=True, sort=False) # concat them
                out_name = f"{school.replace(' ','_').replace('&','and').lower()}_rosters.csv" # filename
                school_df.to_csv(out_name, index=False) # save to CSV
                print(f"  Saved per-school CSV: {out_name}") # log save
                all_frames.append(school_df) # add to all schools list
            else: # no data for this school
                print("  (No rows for this school.)") # log no data

    finally: # ensure driver quits
        driver.quit() # quit the browser

    if not all_frames: # if no data at all
        print("[!] Nothing scraped — check URLs and years.") # log nothing
        return pd.DataFrame() # return empty DataFrame
    return pd.concat(all_frames, ignore_index=True, sort=False) # return all data concatenated



In [ ]:
# ===== 6) RUN =====
master_df = scrape_schools_years_desc_autostop(SCHOOLS, YEARS) # -> pd.DataFrame

print("\n--- SUMMARY ---") # print summary
print(master_df.shape) # print shape
if not master_df.empty: # if we have data
    print(master_df.groupby(["school","year"]).size().sort_index()) # print group sizes
    master_df.to_csv("all_rosters.csv", index=False) # save to CSV
    print("\nSaved master: all_rosters.csv") # log save
else: # no data scraped
    print("No data scraped.") # log no data




=== Jackson State ===
  Year 2025: https://gojsutigers.com/sports/football/roster/2025?view=2
    -> Got 96 rows.
  Year 2024: https://gojsutigers.com/sports/football/roster/2024?view=2
    -> Got 101 rows.
  Year 2023: https://gojsutigers.com/sports/football/roster/2023?view=2
    -> Got 105 rows.
  Year 2022: https://gojsutigers.com/sports/football/roster/2022?view=2
    -> Got 95 rows.
  Year 2021: https://gojsutigers.com/sports/football/roster/2021?view=2
    -> Skipped (empty or <= 1 row). [1 empty in a row]
  Year 2020: https://gojsutigers.com/sports/football/roster/2020?view=2
    -> Skipped (empty or <= 1 row). [2 empty in a row]
  Year 2019: https://gojsutigers.com/sports/football/roster/2019?view=2
    -> Got 83 rows.
  Year 2018: https://gojsutigers.com/sports/football/roster/2018?view=2
    -> Got 90 rows.
  Year 2017: https://gojsutigers.com/sports/football/roster/2017?view=2
    -> Skipped (empty or <= 1 row). [1 empty in a row]
  Year 2016: https://gojsutigers.com/sport

In [ ]:
master_df





NameError: name 'master_df' is not defined